# Phase 2: Feature Engineering & Embeddings

This notebook demonstrates the feature engineering pipeline including:
- Sentence-Transformer embeddings generation
- FAISS indexing for similarity search
- Feature extraction and engineering
- Match score computation
- Job recommendations for resumes

## 1. Project Setup & Imports

In [ ]:
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python version: {sys.version}")

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Tuple
import warnings

warnings.filterwarnings('ignore')

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully")

In [ ]:
# Import project modules
from config.config import (
    JOBS_CSV_PATH, RESUMES_CSV_PATH,
    PROCESSED_JOBS_PATH, PROCESSED_RESUMES_PATH,
    JOBS_EMBEDDINGS_PATH, RESUMES_EMBEDDINGS_PATH,
    DATA_CLEANING_CONFIG, FEATURE_ENGINEERING_CONFIG,
    MODEL_CONFIG
)
from src.data_loader import DataLoader
from src.data_cleaner import DataCleaner, run_full_pipeline
from src.embeddings import EmbeddingsGenerator, FAISSIndex, generate_all_embeddings
from src.feature_engineering import FeatureEngineer, engineer_all_features

print("Project modules imported successfully")

## 2. Data Loading & Preprocessing

In [ ]:
# Load and clean data
print("Loading raw data...")

loader = DataLoader(str(JOBS_CSV_PATH), str(RESUMES_CSV_PATH))
jobs_df, resumes_df = loader.load_data()

print(f"\nRaw data shape:")
print(f"  Jobs: {jobs_df.shape}")
print(f"  Resumes: {resumes_df.shape}")

In [ ]:
# Merge configs and run cleaning pipeline
merged_config = {**DATA_CLEANING_CONFIG, **MODEL_CONFIG}

print("Running data cleaning pipeline...")
cleaned_jobs, cleaned_resumes, train_jobs, test_jobs = run_full_pipeline(
    jobs_df,
    resumes_df,
    merged_config
)

print(f"\nCleaned data shape:")
print(f"  Jobs: {cleaned_jobs.shape}")
print(f"  Resumes: {cleaned_resumes.shape}")
print(f"  Train jobs: {train_jobs.shape}")
print(f"  Test jobs: {test_jobs.shape}")

In [ ]:
# Display cleaned data sample
print("Sample cleaned job record:")
print(cleaned_jobs.iloc[0])

print("\n" + "="*80)
print("\nSample cleaned resume record:")
print(cleaned_resumes.iloc[0])

## 3. Embeddings Generation

In [ ]:
# Generate embeddings
print("Generating embeddings (this may take a few minutes)...\n")

embeddings_result = generate_all_embeddings(
    cleaned_jobs,
    cleaned_resumes,
    FEATURE_ENGINEERING_CONFIG,
    output_dir=PROCESSED_JOBS_PATH.parent
)

generator = embeddings_result['generator']
job_embeddings = embeddings_result['job_embeddings']
resume_embeddings = embeddings_result['resume_embeddings']
faiss_index = embeddings_result['faiss_index']

print(f"\nEmbeddings generated:")
print(f"  Job embeddings shape: {job_embeddings.shape}")
print(f"  Resume embeddings shape: {resume_embeddings.shape}")
print(f"  Embedding dimension: {FEATURE_ENGINEERING_CONFIG['EMBEDDING_DIM']}")

In [ ]:
# Analyze embedding statistics
print("Embedding Statistics\n")

print("Job Embeddings:")
print(f"  Mean: {job_embeddings.mean():.4f}")
print(f"  Std: {job_embeddings.std():.4f}")
print(f"  Min: {job_embeddings.min():.4f}")
print(f"  Max: {job_embeddings.max():.4f}")

print("\nResume Embeddings:")
print(f"  Mean: {resume_embeddings.mean():.4f}")
print(f"  Std: {resume_embeddings.std():.4f}")
print(f"  Min: {resume_embeddings.min():.4f}")
print(f"  Max: {resume_embeddings.max():.4f}")

## 4. Feature Engineering

In [ ]:
# Engineer features
print("Engineering features for all job-resume pairs...\n")

features_result = engineer_all_features(
    cleaned_jobs,
    cleaned_resumes,
    merged_config,
    job_embeddings,
    resume_embeddings
)

features_df = features_result['features_df']
feature_matrix = features_result['feature_matrix']
feature_engineer = features_result['engineer']

print(f"\nFeatures dataframe shape: {features_df.shape}")

In [ ]:
# Display feature columns
print("Feature columns:")
print(features_df.columns.tolist())

print("\nSample features for first pair:")
print(features_df.iloc[0])

In [ ]:
# Feature statistics
feature_cols = ['skill_overlap', 'education_alignment', 'experience_alignment',
                'salary_compatibility', 'semantic_similarity', 'matching_skills_count']

print("Feature Statistics:\n")
print(features_df[feature_cols + ['match_score']].describe())

## 5. Feature Distribution Analysis

In [ ]:
# Plot feature distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, col in enumerate(feature_cols):
    axes[idx].hist(features_df[col], bins=30, edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'Distribution of {col}')
    axes[idx].set_xlabel('Score')
    axes[idx].set_ylabel('Frequency')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Feature distributions plotted")

In [ ]:
# Match score distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
ax1.hist(features_df['match_score'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
ax1.set_title('Match Score Distribution')
ax1.set_xlabel('Match Score')
ax1.set_ylabel('Frequency')
ax1.grid(True, alpha=0.3)

# Box plot
ax2.boxplot(features_df['match_score'], vert=True)
ax2.set_title('Match Score Box Plot')
ax2.set_ylabel('Match Score')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Match Score Range: {features_df['match_score'].min():.3f} to {features_df['match_score'].max():.3f}")

## 6. Feature Correlations

In [ ]:
# Correlation matrix
corr_cols = feature_cols + ['match_score']
correlation_matrix = features_df[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, cbar_kws={'label': 'Correlation'})
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

print("Correlation with match_score:")
print(correlation_matrix['match_score'].sort_values(ascending=False))

## 7. FAISS Similarity Search

In [ ]:
# Test FAISS similarity search
print("Testing FAISS similarity search...\n")

# Take first resume and find similar jobs
test_resume_idx = 0
test_query = resume_embeddings[test_resume_idx:test_resume_idx+1]

distances, indices = faiss_index.search(test_query, k=10)

print(f"Top 10 similar jobs to resume {test_resume_idx}:\n")

for rank, (dist, job_idx) in enumerate(zip(distances[0], indices[0]), 1):
    job = cleaned_jobs.iloc[job_idx]
    similarity = 1 / (1 + dist)  # Convert L2 distance to similarity
    print(f"{rank}. Job: {job['job_title'][:50]}")
    print(f"   Similarity Score: {similarity:.4f}")
    print(f"   Distance: {dist:.4f}\n")

## 8. Top Job Recommendations for Sample Resumes

In [ ]:
# Get top recommendations for multiple resumes
def get_top_recommendations(resume_idx: int, n_recommendations: int = 10) -> pd.DataFrame:
    """
    Get top N job recommendations for a resume.
    """
    # Get features for this resume
    resume_features = features_df[features_df['resume_idx'] == resume_idx].copy()
    
    # Sort by match score
    top_matches = resume_features.nlargest(n_recommendations, 'match_score')
    
    # Add job details
    results = []
    for _, match in top_matches.iterrows():
        job_idx = int(match['job_idx'])
        job = cleaned_jobs.iloc[job_idx]
        results.append({
            'rank': len(results) + 1,
            'job_title': job['job_title'],
            'company': job.get('company', 'N/A')[:30],
            'match_score': f"{match['match_score']:.3f}",
            'skill_overlap': f"{match['skill_overlap']:.1%}",
            'experience_alignment': f"{match['experience_alignment']:.1%}",
            'education_alignment': f"{match['education_alignment']:.1%}"
        })
    
    return pd.DataFrame(results)

# Show recommendations for first 3 resumes
for resume_idx in range(min(3, len(cleaned_resumes))):
    resume = cleaned_resumes.iloc[resume_idx]
    print(f"\n{'='*80}")
    print(f"Recommendations for Resume {resume_idx}: {resume.get('Name', 'Unknown')}")
    print(f"Role: {resume.get('Job Role', 'N/A')} | Experience: {resume.get('Experience (Years)', 0)} years")
    print(f"{'='*80}\n")
    
    recommendations = get_top_recommendations(resume_idx, n_recommendations=10)
    print(recommendations.to_string(index=False))

## 9. Feature Importance Analysis

In [ ]:
# Analyze feature importance by correlation with match score
feature_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Correlation': [features_df[[col, 'match_score']].corr().iloc[0, 1] for col in feature_cols]
}).sort_values('Correlation', ascending=True)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(feature_importance['Feature'], feature_importance['Correlation'])

# Color bars
for bar, val in zip(bars, feature_importance['Correlation']):
    bar.set_color('green' if val > 0 else 'red')

ax.set_xlabel('Correlation with Match Score')
ax.set_title('Feature Importance (Correlation Analysis)')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("\nFeature Importance:")
print(feature_importance.to_string(index=False))

## 10. Summary & Key Metrics

In [ ]:
# Summary statistics
print("\n" + "="*80)
print("PHASE 2: FEATURE ENGINEERING - SUMMARY")
print("="*80)

print(f"\nData Summary:")
print(f"  Cleaned Jobs: {len(cleaned_jobs)}")
print(f"  Resumes: {len(cleaned_resumes)}")
print(f"  Total Job-Resume Pairs: {len(features_df):,}")

print(f"\nEmbeddings:")
print(f"  Model: {FEATURE_ENGINEERING_CONFIG['EMBEDDING_MODEL']}")
print(f"  Dimension: {FEATURE_ENGINEERING_CONFIG['EMBEDDING_DIM']}")
print(f"  Job Embeddings Shape: {job_embeddings.shape}")
print(f"  Resume Embeddings Shape: {resume_embeddings.shape}")

print(f"\nEngineered Features:")
print(f"  Total Features: {len(feature_cols)}")
print(f"  Feature Columns: {', '.join(feature_cols)}")

print(f"\nMatch Score Statistics:")
print(f"  Mean: {features_df['match_score'].mean():.3f}")
print(f"  Median: {features_df['match_score'].median():.3f}")
print(f"  Std Dev: {features_df['match_score'].std():.3f}")
print(f"  Min: {features_df['match_score'].min():.3f}")
print(f"  Max: {features_df['match_score'].max():.3f}")
print(f"  25th Percentile: {features_df['match_score'].quantile(0.25):.3f}")
print(f"  75th Percentile: {features_df['match_score'].quantile(0.75):.3f}")

print(f"\nGood Matches (score >= 0.7): {(features_df['match_score'] >= 0.7).sum()} ({(features_df['match_score'] >= 0.7).sum()/len(features_df)*100:.1f}%)")
print(f"Excellent Matches (score >= 0.8): {(features_df['match_score'] >= 0.8).sum()} ({(features_df['match_score'] >= 0.8).sum()/len(features_df)*100:.1f}%)")

print(f"\nTop Feature Correlations with Match Score:")
for feature, corr in correlation_matrix['match_score'].sort_values(ascending=False)[1:4].items():
    print(f"  {feature}: {corr:.3f}")

In [ ]:
# Save features for next phase
print("\nSaving engineered features and embeddings...")

features_df.to_csv(PROCESSED_JOBS_PATH.parent / 'features_engineered.csv', index=False)
print(f"Saved features to: {PROCESSED_JOBS_PATH.parent / 'features_engineered.csv'}")

print("\nPhase 2: Feature Engineering Complete!")
print("Ready for Phase 3: Model Training")